# 6b · Event-centric reification with Gemini  (per article)

The **reification twin** of `6_relation_extraction.ipynb`. Instead of stating facts as binary
`[subject, property, object]` triples, we gather the participants of *one* news fact under *one*
**hub node** via an **open, extensible set of semantic roles** (a seed vocabulary the model may grow,
same philosophy as nb6/nb7 properties). See `EVENT_REIFICATION_DESIGN.md` for the full
spec and motivation.

**Three hub kinds** (schema.org-first): **Event** (`Event` subtree — something happens),
**Observation** (`Observation` — a stated statistic/measurement/change), **Claim**
(`Claim`/`Statement` — an attributed statement). **Hybrid rule:** only these are reified; genuinely
binary facts (`Paris partOf France`) stay as plain edges in `binary_relations`.

Reads the **same** final entity nodes as nb6 — `typed_deduped_entities_ext_consolidated.json` if
present, else `typed_deduped_entities_ext.json` (both from nb4) — id-stamped `E1, E2, …` so hubs can
point at participants unambiguously. Writes `results/<ARTICLE_ID>/reified_graph.json`, leaving the
binary path (`6_relation_extraction` → `7`) untouched so the two graphs stay comparable.

Set `ARTICLE_ID` for the single-article demo cell, or list ids in `SELECTED_ARTICLES` and run the
**batch** cell.

In [26]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR  = Path("results")            # reads results/<id>/typed_deduped_entities_ext*.json (nb4)
DATASET_DIR  = Path("dataset/articles")   # reads <id>.txt for the REFERENCE_DATE and evidence spans
GEMINI_MODEL = "gemini-2.5-flash"

ARTICLE_ID   = "art1"                      # single article used by the demo cell below

# Which articles the BATCH cell processes, in one run.
#   None -> every article that has an entity file (typed_deduped_entities_ext*.json), sorted.
#   list -> exactly these ids, in this order, e.g. ["art1", "art2", "art20"].
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

# Which per-article entity file to use as participants. Consolidated (post global re-consolidation in
# nb4) preferred when present; otherwise the raw incremental one.  (Same as nb6.)
ENTITY_FILES = ["typed_deduped_entities_ext_consolidated.json", "typed_deduped_entities_ext.json"]

OUTPUT_NAME  = "reified_graph.json"        # written to results/<id>/

def entity_path(aid):
    for name in ENTITY_FILES:
        p = RESULTS_DIR / aid / name
        if p.exists():
            return p
    return None

_available = sorted({p.parent.name for p in RESULTS_DIR.glob("*/typed_deduped_entities_ext*.json")})
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    assert not missing, f"no entity file for: {missing}"
    ARTICLE_IDS = list(SELECTED_ARTICLES)

print(f"{len(_available)} article(s) have entity files; this run will process {ARTICLE_IDS}")

4 article(s) have entity files; this run will process ['art1', 'art2', 'art4', 'art5']


In [27]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

# Reified graphs are large (many hubs x roles + the binary-relations pass), so the JSON output can be
# long. Raise this if you still hit truncation on very table-heavy articles.
MAX_OUTPUT_TOKENS = 32768

def _loads_lenient(text, resp=None):
    """Parse Gemini's JSON, tolerating code fences / trailing commas, with a CLEAR message on truncation."""
    t = (text or "").strip()
    if t.startswith("```"):                                   # strip ```json ... ``` fences if present
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t)
        t = re.sub(r"\n?```$", "", t).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    t2 = re.sub(r",(\s*[}\]])", r"\1", t)                     # drop trailing commas before } or ]
    try:
        return json.loads(t2)
    except json.JSONDecodeError as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        raise ValueError(
            f"Gemini returned unparseable JSON (finish_reason={fr}; {len(t)} chars). "
            f"If finish_reason is MAX_TOKENS the output was TRUNCATED mid-JSON — raise MAX_OUTPUT_TOKENS "
            f"(now {MAX_OUTPUT_TOKENS}) and/or split the article. json error: {e}\n"
            f"...last 300 chars of output:\n{t[-300:]}"
        ) from None

def gemini_json(system_prompt, user_message, model=None, temperature=0, max_output_tokens=None):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
            max_output_tokens=max_output_tokens or MAX_OUTPUT_TOKENS,
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return _loads_lenient(resp.text, resp)

Note: you may need to restart the kernel to use updated packages.


## 1. Load the article's entities (participants)

Identical to nb6 so the reified graph and the binary graph draw from the **same** node set. Each
entity node from nb4 becomes an id-stamped **entity card** (`E1, E2, …`) the model may reference from
a role's `value_ref`.

In [28]:
def load_entities(aid):
    """Read the article's final entity nodes (nb4) and build id-stamped participant cards.
    Returns (basename, cards) where each card is {id, text, type, aliases}."""
    p = entity_path(aid)
    assert p is not None, f"no entity file for {aid} (expected one of {ENTITY_FILES} under results/{aid}/)"
    data = json.loads(p.read_text())
    cards = []
    for i, e in enumerate(data.get("entities", []), 1):
        cards.append({
            "id":      f"E{i}",
            "text":    e.get("canonical_text"),
            "type":    e.get("chosen_type"),
            "aliases": e.get("aliases", []),
        })
    return p.name, cards

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"

_fname, _cards = load_entities(ARTICLE_ID)
print(f"{ARTICLE_ID}: {len(_cards)} entities from {_fname}")
for c in _cards[:8]:
    print(f"  {c['id']:<4} {c['text']!r:<28} :: {c['type']}")

art1: 103 entities from typed_deduped_entities_ext_consolidated.json
  E1   'France'                     :: Country
  E2   'the UK'                     :: Country
  E3   'Portugal'                   :: Country
  E4   'Spain'                      :: Country
  E5   'Belgium'                    :: Country
  E6   'the Netherlands'            :: Country
  E7   'the country'                :: Country
  E8   '2,025 excess deaths'        :: QuantitativeValue


## 2. The seed role vocabulary (generic backbone, open / extensible)

A small **domain-neutral** starter set of thematic roles (who/what/where/when/why/how) a hub may use
to point at participants (§3 of the design doc) — **not** a closed list. Domain specifics
(`changeRate`, `reportedOn`, `evacuees`, …) are intentionally left out: the model reuses a seed role
when one fits and otherwise **invents** a new atomic role (with a definition, an optional `parent`
pointing at the closest seed role, and `role_source:"new"`) — exactly like the open-vocabulary
`property` mechanics of the binary path. Invented roles are collected per article into `new_roles`
so a later pass can consolidate the growing role vocabulary (mirroring
`relation_vocabulary_consolidated.json`), and common ones can graduate into the seed.

In [29]:
# Seed (preferred, NOT closed) role set — a small DOMAIN-NEUTRAL backbone of thematic roles
# (who/what/where/when/why/how). Domain specifics (changeRate, reportedOn, evacuees, burnedArea, ...)
# are intentionally omitted so the model INVENTS them, usually parented to a seed role. Edit to grow.
SEED_ROLES = [
    {"role": "agent",       "schema_property": "agent",       "definition": "Who/what brings the hub about — the doer/actor."},
    {"role": "affected",    "schema_property": "ex:affected",  "definition": "Who/what undergoes or is affected (patient / theme / experiencer)."},
    {"role": "about",       "schema_property": "about",        "definition": "What the hub concerns or measures — its topic/subject."},
    {"role": "location",    "schema_property": "location",     "definition": "Where it happens or is observed."},
    {"role": "time",        "schema_property": "startDate",    "definition": "When — a point or interval; value may be a Date/Duration node or a literal."},
    {"role": "value",       "schema_property": "value",        "definition": "How much — a quantity, measurement, or magnitude."},
    {"role": "cause",       "schema_property": "ex:cause",     "definition": "Why — the event/condition that brought it about."},
    {"role": "result",      "schema_property": "ex:result",    "definition": "The outcome or effect it produces."},
    {"role": "instrument",  "schema_property": "ex:instrument","definition": "By what means or method it is done."},
    {"role": "source",      "schema_property": "source",       "definition": "Origin — where it comes from, or who states/reports it."},
    {"role": "participant", "schema_property": "participant",  "definition": "Any other involved entity not captured by a more specific role."},
    {"role": "partOf",      "schema_property": "superEvent",   "definition": "The larger event/whole this hub belongs to."},
]

SEED_ROLE_NAMES = [r["role"] for r in SEED_ROLES]
print(len(SEED_ROLE_NAMES), "seed roles (extensible):", ", ".join(SEED_ROLE_NAMES))

12 seed roles (extensible): agent, affected, about, location, time, value, cause, result, instrument, source, participant, partOf


## 3. Event-centric reification

One Gemini call per article. The model receives the TEXT, the REFERENCE_DATE, the entity cards, the
three hub kinds, and the seed role vocabulary (extensible), and returns hubs (Event / Observation / Claim) plus the
leftover genuinely-binary facts. A role's value is an entity id (`value_ref`), another hub id
(`value_ref_hub`), or a literal (`value_text`) — exactly one primary per instance.

In [30]:
REIFY_SYSTEM_PROMPT = """You are an expert event-centric information extraction system.
You are given a TEXT (a news article), its REFERENCE_DATE (publication date, ISO-8601), a list of ENTITIES already extracted from it (each with an id, surface text, type, aliases), and a SEED_ROLE vocabulary (preferred, but NOT closed — you may extend it).

GOAL: Represent the news as an event-centric graph. Rather than binary triples, gather the participants of ONE fact under ONE hub node, connected by roles.

HUB KINDS (create hubs only of these kinds):
- "Event": something that happens (heatwave, wildfire, evacuation, forecast, alert declaration). Type it with a schema.org Event subtype when one fits (e.g. WeatherEvent); otherwise mint a concise type and set type_source "ex".
- "Observation": a stated statistic, measurement, or quantified change (e.g. excess-death counts, temperatures, hectares burned, percentage rises). Base type "Observation".
- "Claim": a statement explicitly attributed to a source ("X said Y", "the ministry called it unprecedented"). Base type "Claim". Give it an attributedTo role and an about role.

RULES:
1. Identify the article's MAIN event(s) — the central happening(s) the news is about — and set "is_main": true on them. Hang sub-events / observations / claims onto a main event with the "partOfEvent" or "cause" role so the graph has a clear backbone.
2. Each hub has: id (EV1, OBS1, CL1, ... unique within the article), hub_kind, type, type_source ("schema.org" or "ex"), a short human-readable label, is_main (bool), roles[], evidence[] (verbatim spans), confidence (0-1).
3. Each role instance has a "role" plus role provenance and EXACTLY ONE primary value:
     "role": prefer a role from SEED_ROLES; if none fits, INVENT a concise new one.
     "role_source": "seed" (reused a seed role) | "schema.org" (a seed role that maps to a schema.org property) | "new" (invented).
     If "new": "definition" is REQUIRED (one sentence) and "parent" is OPTIONAL (an existing/seed role it specializes, for hierarchy).
     Value — pick exactly one primary:
       "value_ref": "<entity id from ENTITIES>"   — when the participant is one of the given entities, OR
       "value_ref_hub": "<another hub id>"          — when it is another hub (e.g. cause -> an Event), OR
       "value_text": "<literal>"                    — when it is a literal not worth a node (a unit, a raw percentage, a reported date, a source name not in ENTITIES).
   You MAY add "value_text" alongside value_ref as a gloss, but pick one primary.
   INVENTION DISCIPLINE: the SEED_ROLES are deliberately generic, so you WILL need to invent domain-specific roles (e.g. "changeRate", "reportedOn", "evacuees", "burnedArea") — set each one's "parent" to the closest seed role. Invented roles MUST be ATOMIC and reusable — one participant slot. NEVER pack context into the role name (BAD: "notedIncreaseInDeathsForAgeGroup"); time/place/value/cause each get their OWN role edge — that is the point of the hub.
4. Prefer value_ref to existing ENTITIES; do not duplicate an entity as a literal if its id is available. Do not relate a hub to itself.
5. BINARY-RELATIONS PASS (MANDATORY — do NOT skip, do NOT leave empty on a normal article): AFTER the hubs, make a SEPARATE sweep over the ENTITIES and record every stable entity<->entity fact in binary_relations — EVEN IF those entities already appear as participants inside a hub. These are plain edges, NOT wrapped in hubs and NOT hub roles. You MUST capture at least these kinds when the TEXT supports them:
     - geographic containment / part-of: a place inside a larger place ("Paris" containedInPlace "France"; "southern Britain" isPartOf "the UK").
     - organizational membership / affiliation: an agency, ministry, or person belonging to an org/country ("Public Health France" memberOf "France"; "Justin Rowlatt" worksFor "BBC").
     - role / title: a person holding an office of a country/org ("Rist" ministerOf "France").
     - ownership / identity: copyright holder, publisher, sameAs.
   Each entry: {"subject_id","property","object_id","property_source","evidence"} — property is a concise label (REUSE a schema.org property such as containedInPlace / memberOf / worksFor / copyrightHolder when one fits, else mint a camelCase one); property_source is "schema.org" or "ex". Orient subject -> property -> object correctly. Do NOT force these facts into hub roles, and do NOT omit them just because the entities are already used in a hub.
6. Extract only what the TEXT explicitly states. Use REFERENCE_DATE to resolve relative dates in value_text where possible.

OUTPUT strict JSON:
{
  "hubs": [ {"id","hub_kind","type","type_source","label","is_main",
             "roles":[{"role","role_source","parent"?,"definition"?,"value_ref"?,"value_ref_hub"?,"value_text"?}],
             "evidence":[...],"confidence"} ],
  "binary_relations": [ {"subject_id","property","object_id","property_source","evidence"} ]
}
"""

def _fmt_entities(cards):
    return "\n".join(
        f'{c["id"]}\t{c["text"]}\t({c["type"]})' + (f'\taliases: {c["aliases"]}' if c["aliases"] else "")
        for c in cards
    )

def _fmt_seed_roles():
    return "\n".join(f'- {r["role"]} ({r["schema_property"]}): {r["definition"]}' for r in SEED_ROLES)

def _collect_new_roles(hubs):
    """Gather roles the model invented (role_source == 'new') for cross-article consolidation."""
    seen = {}
    for h in hubs:
        for r in h.get("roles", []):
            if r.get("role_source") == "new" or (r.get("role") not in SEED_ROLE_NAMES):
                k = r["role"]
                e = seen.setdefault(k, {"role": k, "parent": r.get("parent"),
                                        "definition": r.get("definition"), "count": 0})
                e["count"] += 1
    return sorted(seen.values(), key=lambda e: -e["count"])

def reify_article(aid, write=True):
    """Run event-centric reification for one article -> results/<id>/reified_graph.json."""
    fname, cards = load_entities(aid)
    text = (DATASET_DIR / f"{aid}.txt").read_text()
    ref_date = _reference_date(text)

    user_message = (
        f"REFERENCE_DATE: {ref_date}\n\n"
        f"SEED_ROLES (preferred, extensible — invent new atomic roles when none fit):\n{_fmt_seed_roles()}\n\n"
        f"ENTITIES (id\ttext\t(type)):\n{_fmt_entities(cards)}\n\n"
        f"TEXT:\n{text}"
    )
    out = gemini_json(REIFY_SYSTEM_PROMPT, user_message)
    hubs = out.get("hubs", [])

    graph = {
        "article_id":      aid,
        "reference_date":  ref_date,
        "entity_file":     fname,
        "entities":        cards,
        "hubs":            hubs,
        "binary_relations": out.get("binary_relations", []),
        "seed_roles":      SEED_ROLES,
        "new_roles":       _collect_new_roles(hubs),
    }
    if write:
        p = RESULTS_DIR / aid / OUTPUT_NAME
        p.write_text(json.dumps(graph, indent=2, ensure_ascii=False))
        print(f"  wrote {p}  ({len(hubs)} hubs, {len(graph['binary_relations'])} binary, "
              f"{len(graph['new_roles'])} new roles)  [{last_latency_s:.1f}s]")
    return graph

## 4. Inspect the demo article

In [31]:
# helper to resolve a role's value to readable text
def _resolve(role, cards_by_id, hubs_by_id):
    if role.get("value_ref"):     return cards_by_id.get(role["value_ref"], {}).get("text", role["value_ref"])
    if role.get("value_ref_hub"): return f'[{hubs_by_id.get(role["value_ref_hub"], {}).get("label", role["value_ref_hub"])}]'
    return role.get("value_text", "?")

# g = reify_article(ARTICLE_ID)
# cards_by_id = {c["id"]: c for c in g["entities"]}
# hubs_by_id  = {h["id"]: h for h in g["hubs"]}

# print("-" * 78)
# for h in g["hubs"]:
#     star = " *MAIN*" if h.get("is_main") else ""
#     print(f'{h["id"]}  {h["hub_kind"]}<{h["type"]}>  "{h["label"]}"{star}   (conf {h.get("confidence")})')
#     for r in h["roles"]:
#         mark = " [NEW]" if (r.get("role_source") == "new" or r.get("role") not in SEED_ROLE_NAMES) else ""
#         print(f'      {r["role"]:<18}{mark:<6} -> {_resolve(r, cards_by_id, hubs_by_id)!r}')
# print("-" * 78)
# print(f'{len(g["binary_relations"])} leftover binary relations:')
# for b in g["binary_relations"][:15]:
#     s = cards_by_id.get(b["subject_id"], {}).get("text", b["subject_id"])
#     o = cards_by_id.get(b["object_id"], {}).get("text", b["object_id"])
#     print(f'   {s!r} --{b["property"]}--> {o!r}')
# print("-" * 78)
# print(f'invented (new) roles this article: {[r["role"] for r in g["new_roles"]]}')

## 5. Batch — process the selected articles (`ARTICLE_IDS`)

In [ ]:
# Runs reification for each article in ARTICLE_IDS (choose them via SELECTED_ARTICLES in the config
# cell). Re-runs overwrite existing outputs. The binary path (nb6/nb7) is untouched.
for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    reify_article(aid)
print(f"\ndone -- reified {len(ARTICLE_IDS)} article(s) -> results/<id>/{OUTPUT_NAME}")

============================================================ art1
